<a href="https://colab.research.google.com/github/gerney177/5th-diet-ai/blob/data-processing/%ED%96%89%EC%A0%95%EB%B2%95_LLM_%EB%8D%B0%EC%9D%B4%ED%84%B0_%EC%A0%84%EC%B2%98%EB%A6%AC_%EB%B0%8F_JSONL_%EC%83%9D%EC%84%B1_%EC%8B%9C%EC%8A%A4%ED%85%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
"""
행정법 LLM 데이터 전처리 및 JSONL 생성 시스템
Google Colab 환경에서 실행
"""

# 필요한 라이브러리 설치
!pip install openai tqdm

import os
import json
import re
from pathlib import Path
from tqdm import tqdm
from openai import OpenAI
from google.colab import drive

# Google Drive 마운트
drive.mount('/content/drive')

# OpenAI API 키 설정 (사용자가 입력)
OPENAI_API_KEY = "your-openai-api-key-here"  # OpenAI API 키를 여기에 입력
client = OpenAI(api_key=OPENAI_API_KEY)

MODEL_NAME = "gpt-4o-mini"

# 경로 설정 - 실제 폴더 구조에 맞게 수정
DRIVE_BASE = "/content/drive/MyDrive"
OUTPUT_PATH = f"{DRIVE_BASE}/processed_data"

# QA 데이터 폴더들
QA_FOLDERS = [
    "VL_결정례_QA",
    "VL_법령_QA",
    "VL_판결문_QA",
    "VL_해석례_QA"
]

# SUM 데이터 폴더들
SUM_FOLDERS = [
    "VL_결정례_SUM",
    "VL_법령_SUM",
    "VL_판결문_SUM",
    "VL_해석례_SUM"
]

# 출력 디렉토리 생성
os.makedirs(OUTPUT_PATH, exist_ok=True)

# 1. JSON 데이터 로드 및 파싱
def load_json_files(folder_names, base_path=DRIVE_BASE):
    """여러 폴더에서 모든 JSON 파일을 로드"""
    all_data = []

    for folder in folder_names:
        folder_path = os.path.join(base_path, folder)

        if not os.path.exists(folder_path):
            print(f"⚠️ 폴더를 찾을 수 없습니다: {folder_path}")
            continue

        print(f"📁 {folder} 폴더 처리 중...")

        # 재귀적으로 JSON 파일 찾기
        json_files = list(Path(folder_path).rglob("*.json"))
        print(f"   찾은 JSON 파일: {len(json_files)}개")

        for json_file in json_files:
            try:
                with open(json_file, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    data['_filename'] = json_file.name
                    data['_folder'] = folder
                    all_data.append(data)
            except Exception as e:
                print(f"   오류: {json_file.name} - {e}")

    return all_data

# 2. GPT API를 활용한 데이터 재구성 프롬프트
SYSTEM_PROMPT = """너는 행정법 전문 AI 교육관이야.
아래 제공되는 [법령 원문]과 [기존 질의응답]을 참고해서, 로컬 AI 모델을 학습시키기 위한 최적의 데이터를 만들어줘.

**요구사항:**
1. 답변은 반드시 '관련 법령 몇 조'에 근거하는지 명시할 것
2. 답변의 논리 전개 과정을 '판단 근거 → 상세 설명 → 결론' 순으로 구성할 것
3. 기계적인 말투를 자연스럽고 전문적인 어조로 다듬을 것
4. 법리 해석 과정을 명확히 제시할 것
5. 15어절 이상의 서술형으로 작성할 것

**중요: 반드시 유효한 JSON 형식으로만 응답할 것. 어떠한 추가 설명이나 마크다운도 포함하지 말 것.**

**출력 형식:**
{
  "instruction": "질문 내용",
  "input": "",
  "output": "근거 법령을 포함한 상세 답변 (판단 근거 → 상세 설명 → 결론)"
}
"""

def clean_json_response(content):
    """JSON 응답에서 불필요한 부분 제거 및 정리"""
    # 마크다운 코드 블록 제거
    if "```json" in content:
        content = re.sub(r'```json\s*', '', content)
        content = re.sub(r'```\s*$', '', content)
    elif "```" in content:
        content = re.sub(r'```\s*', '', content)

    # 앞뒤 공백 제거
    content = content.strip()

    # JSON 객체만 추출 (중괄호로 시작하고 끝나는 부분)
    json_match = re.search(r'\{.*\}', content, re.DOTALL)
    if json_match:
        content = json_match.group(0)

    return content

def create_refined_data(original_data, retry_count=0, max_retries=2):
    """GPT API를 사용해 데이터 재구성 (재시도 로직 포함)"""

    # 원본 데이터에서 필요한 정보 추출
    info = original_data.get('info', {})
    label = original_data.get('label', {})

    agenda = info.get('agenda', '')
    input_question = label.get('input', '')
    original_output = label.get('output', '')

    # 데이터가 없으면 스킵
    if not input_question or not original_output:
        return None

    # 텍스트가 너무 길면 자르기 (API 한도 방지)
    if len(agenda) > 2000:
        agenda = agenda[:2000] + "..."
    if len(original_output) > 1000:
        original_output = original_output[:1000] + "..."

    # GPT에게 전달할 사용자 프롬프트
    user_prompt = f"""
[사건/법령 개요]
{agenda}

[기존 질문]
{input_question}

[기존 답변]
{original_output}

위 내용을 바탕으로:
1. 관련 법령 조항을 명확히 밝히고
2. 판단 근거 → 법리 해석 → 상세 설명 → 결론 순서로 논리적으로 재구성하며
3. 자연스럽고 전문적인 어조로 다듬어진 답변을 만들어줘.

반드시 유효한 JSON 형식으로만 응답하고, 다른 설명이나 마크다운은 절대 포함하지 마.
"""

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.5,
            max_tokens=1500
        )

        content = response.choices[0].message.content.strip()

        # 응답이 비어있으면 스킵
        if not content:
            print(f"⚠️  빈 응답 수신")
            return None

        # JSON 정리
        content = clean_json_response(content)

        # JSON 파싱 시도
        try:
            refined_data = json.loads(content)
        except json.JSONDecodeError as je:
            # JSON 파싱 실패 시 재시도
            if retry_count < max_retries:
                print(f"🔄 JSON 파싱 실패, 재시도 {retry_count + 1}/{max_retries}")
                return create_refined_data(original_data, retry_count + 1, max_retries)
            else:
                print(f"❌ JSON 파싱 최종 실패: {str(je)[:100]}")
                print(f"   응답 내용 (처음 200자): {content[:200]}")
                return None

        # 필수 필드 확인
        if 'instruction' not in refined_data or 'output' not in refined_data:
            print(f"⚠️  필수 필드 누락")
            return None

        # input 필드가 없으면 추가
        if 'input' not in refined_data:
            refined_data['input'] = ""

        # 메타데이터 추가
        refined_data['metadata'] = {
            'lawClass': info.get('lawClass', ''),
            'docuType': info.get('DocuType', ''),
            'interpreId': info.get('interpreId', ''),
            'interpreDate': info.get('interpreDate', ''),
            'source_folder': original_data.get('_folder', ''),
            'source_file': original_data.get('_filename', '')
        }

        return refined_data

    except Exception as e:
        print(f"❌ API 오류: {str(e)[:100]}")
        return None

# 3. Batch 처리 함수
def process_batch(data_list, output_file, batch_size=10):
    """데이터를 배치로 처리하고 JSONL로 저장"""

    results = []
    success_count = 0
    error_count = 0

    with open(output_file, 'w', encoding='utf-8') as f:
        for i in tqdm(range(0, len(data_list), batch_size), desc="처리 중"):
            batch = data_list[i:i+batch_size]

            for data in batch:
                refined = create_refined_data(data)
                if refined:
                    # JSONL 형식으로 저장 (한 줄에 하나의 JSON)
                    f.write(json.dumps(refined, ensure_ascii=False) + '\n')
                    results.append(refined)
                    success_count += 1
                else:
                    error_count += 1

            # 진행 상황 출력
            if (i // batch_size) % 10 == 0 and i > 0:
                print(f"\n📊 중간 결과 - 성공: {success_count}, 실패: {error_count}")

    print(f"\n✅ 최종 성공: {success_count}개 | ❌ 최종 실패: {error_count}개")
    return results

# 4. 메인 실행 함수
def main():
    print("="*70)
    print("🚀 행정법 데이터 전처리 시작")
    print("="*70)

    # 1. QA 데이터 처리
    print("\n[1/2] QA 데이터 로드 중...")
    print(f"대상 폴더: {', '.join(QA_FOLDERS)}")
    qa_data = load_json_files(QA_FOLDERS)
    print(f"✅ 총 {len(qa_data)}개의 QA JSON 파일 로드 완료\n")

    if qa_data:
        print("QA 데이터 처리 및 JSONL 생성 중...")
        qa_output = f"{OUTPUT_PATH}/qa_refined.jsonl"
        process_batch(qa_data, qa_output, batch_size=5)
        print(f"💾 QA 데이터 저장 완료: {qa_output}\n")
    else:
        print("⚠️ QA 데이터가 없습니다.\n")

    # 2. SUM 데이터 처리
    print("[2/2] SUM 데이터 로드 중...")
    print(f"대상 폴더: {', '.join(SUM_FOLDERS)}")
    sum_data = load_json_files(SUM_FOLDERS)
    print(f"✅ 총 {len(sum_data)}개의 SUM JSON 파일 로드 완료\n")

    if sum_data:
        print("SUM 데이터 처리 및 JSONL 생성 중...")
        sum_output = f"{OUTPUT_PATH}/sum_refined.jsonl"
        process_batch(sum_data, sum_output, batch_size=5)
        print(f"💾 SUM 데이터 저장 완료: {sum_output}\n")
    else:
        print("⚠️ SUM 데이터가 없습니다.\n")

    # 3. 최종 통계
    print("\n" + "="*70)
    print("🎉 전처리 완료!")
    print("="*70)
    print(f"📂 결과 파일 위치: {OUTPUT_PATH}")

    # 생성된 파일 확인
    if os.path.exists(OUTPUT_PATH):
        output_files = os.listdir(OUTPUT_PATH)
        print(f"\n생성된 파일:")
        for f in output_files:
            file_path = os.path.join(OUTPUT_PATH, f)
            file_size = os.path.getsize(file_path) / 1024 / 1024  # MB
            print(f"  • {f} ({file_size:.2f} MB)")

    print("="*70)

if __name__ == "__main__":
    if OPENAI_API_KEY == "your-openai-api-key-here":
        print("⚠️  코드 상단의 OPENAI_API_KEY 변수에 실제 OpenAI API 키를 입력하세요.")
        print("API 키 발급: https://platform.openai.com/api-keys")
    else:
        main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 행정법 데이터 전처리 시작 (재개 가능 버전)
⚙️  설정: 병렬 쓰레드 20개

[1/2] QA 데이터 로드 중...
📁 VL_결정례_QA 폴더 처리 중...
   찾은 JSON 파일: 646개
📁 VL_법령_QA 폴더 처리 중...
   찾은 JSON 파일: 3982개
📁 VL_판결문_QA 폴더 처리 중...
   찾은 JSON 파일: 1611개
📁 VL_해석례_QA 폴더 처리 중...
   찾은 JSON 파일: 510개
✅ 총 6749개의 QA JSON 파일 로드 완료

QA 데이터 처리 중...
🔄 전체 6749개 중 461개 남음 (이미 처리: 6288개)
🚀 20개 쓰레드로 병렬 처리 시작...


처리 중:   3%|▎         | 15/461 [00:06<01:05,  6.81it/s]


📊 진행: 성공 6300, 실패 0


처리 중:  13%|█▎        | 62/461 [00:22<01:36,  4.15it/s]


📊 진행: 성공 6350, 실패 0


...


📊 진행: 성공 6700, 실패 0


처리 중: 100%|██████████| 461/461 [02:23<00:00,  3.21it/s]



✅ 최종 성공: 6749개 | ❌ 최종 실패: 0개
💾 QA 데이터 저장 완료: /content/drive/MyDrive/processed_data/qa_refined.jsonl

[2/2] SUM 데이터 로드 중...
📁 VL_결정례_SUM 폴더 처리 중...
   찾은 JSON 파일: 620개
⚠️ 폴더를 찾을 수 없습니다: /content/drive/MyDrive/VL_법령_SUM
📁 VL_판결문_SUM 폴더 처리 중...
   찾은 JSON 파일: 2240개
📁 VL_해석례_SUM 폴더 처리 중...
   찾은 JSON 파일: 391개
✅ 총 3251개의 SUM JSON 파일 로드 완료

SUM 데이터 처리 중...
🔄 전체 3251개 중 3251개 남음 (이미 처리: 0개)
🚀 20개 쓰레드로 병렬 처리 시작...


처리 중: 100%|██████████| 3251/3251 [00:00<00:00, 182886.91it/s]


📊 진행: 성공 0, 실패 50

📊 진행: 성공 0, 실패 100

📊 진행: 성공 0, 실패 150

📊 진행: 성공 0, 실패 200

📊 진행: 성공 0, 실패 250

📊 진행: 성공 0, 실패 300

📊 진행: 성공 0, 실패 350

📊 진행: 성공 0, 실패 400

📊 진행: 성공 0, 실패 450

📊 진행: 성공 0, 실패 500

📊 진행: 성공 0, 실패 550

📊 진행: 성공 0, 실패 600

📊 진행: 성공 0, 실패 650

📊 진행: 성공 0, 실패 700

📊 진행: 성공 0, 실패 750

📊 진행: 성공 0, 실패 800

📊 진행: 성공 0, 실패 850

📊 진행: 성공 0, 실패 900

📊 진행: 성공 0, 실패 950

📊 진행: 성공 0, 실패 1000

📊 진행: 성공 0, 실패 1050

📊 진행: 성공 0, 실패 1100

📊 진행: 성공 0, 실패 1150

📊 진행: 성공 0, 실패 1200

📊 진행: 성공 0, 실패 1250

📊 진행: 성공 0, 실패 1300

📊 진행: 성공 0, 실패 1350

📊 진행: 성공 0, 실패 1400

📊 진행: 성공 0, 실패 1450

📊 진행: 성공 0, 실패 1500

📊 진행: 성공 0, 실패 1550

📊 진행: 성공 0, 실패 1600

📊 진행: 성공 0, 실패 1650

📊 진행: 성공 0, 실패 1700

📊 진행: 성공 0, 실패 1750

📊 진행: 성공 0, 실패 1800

📊 진행: 성공 0, 실패 1850

📊 진행: 성공 0, 실패 1900

📊 진행: 성공 0, 실패 1950

📊 진행: 성공 0, 실패 2000

📊 진행: 성공 0, 실패 2050

📊 진행: 성공 0, 실패 2100

📊 진행: 성공 0, 실패 2150

📊 진행: 성공 0, 실패 2200

📊 진행: 성공 0, 실패 2250

📊 진행: 성공 0, 실패 2300

📊 진행: 성공 0, 실패 2350

📊 진행: 성공 0, 실패 2400

📊 진행: 성공 0,